# 5) Forecasts in operations — Exercises

**Goals:** derive the cost-minimising quantile, test a rule of thumb for imbalance costs, find the best schedule under steeper penalties, tune the battery's trading rule, and put a price on a better solar forecast.

The setup cell holds lesson 2's demand forecast `REGRESSION`, its honest 2024 errors `ERRORS_2024`, the 2025 demand `ACTUAL`, the solar data `SOLAR`, the prices `PRICE` and `P2025`, the 2024 hourly average price `HOUR_AVG_2024`, and the lecture's `imbalance_cost` and `battery_profit` (with an adjustable trading margin). Power in MW, prices in €/MWh, money in €.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, market_price, merit_order_price, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)
REGRESSION = smf.ols(FORMULA, df.loc[:"2024"]).fit().predict(honest)                     # lesson 2's forecast
ERRORS_2024 = (df.demand - smf.ols(FORMULA, df.loc["2023"]).fit().predict(honest)).loc["2024"]
ACTUAL = df.demand.loc["2025"]
SOLAR = solar_data(df)
PRICE = market_price(df.demand - SOLAR.solar)
P2025 = PRICE.loc["2025"]
HOUR_AVG_2024 = PRICE.loc["2024"].groupby(PRICE.loc["2024"].index.hour).mean()


def imbalance_cost(schedule, short=60.0, long=40.0, actual=ACTUAL):
    """Imbalance cost (EUR) of buying `schedule` when `actual` is consumed."""
    gap = actual - schedule.loc[actual.index]
    return float((short * gap.clip(lower=0) + long * (-gap).clip(lower=0)).sum())


def battery_profit(signal, actual=P2025, margin=0.0):
    """The lecture's battery: profit (EUR) and trading days of a daily 4-hour arbitrage driven by `signal`.
    It trades only if 0.85 x forecast dear price exceeds the forecast cheap price by more than `margin` (EUR/MWh)."""
    profit, days = 0.0, 0
    for _, day in pd.DataFrame({"signal": signal, "actual": actual}).groupby(actual.index.date):
        order = day.sort_values("signal")
        cheap, dear = order.index[:4], order.index[-4:]
        if 0.85 * day.loc[dear, "signal"].mean() - day.loc[cheap, "signal"].mean() <= margin:
            continue
        days += 1
        profit += 100 * (0.85 * day.loc[dear, "actual"].sum() - day.loc[cheap, "actual"].sum())
    return profit, days

### Warm-ups

**1. The cost-minimising quantile**

Write `best_quantile(short, long)`: the quantile α* a supplier should buy when a shortfall costs `short` and a surplus `long` €/MWh. Check the lecture's case and a few others.

In [2]:
def best_quantile(short: float, long: float) -> float:
    """Newsvendor quantile: the share of outcomes that should lie below the amount bought."""
    ### BEGIN SOLUTION
    return short / (short + long)
    ### END SOLUTION

In [3]:
assert np.isclose(best_quantile(60, 40), 0.6)
assert np.isclose(best_quantile(40, 40), 0.5), "symmetric costs: buy the median"
assert np.isclose(best_quantile(150, 40), 150 / 190)
assert best_quantile(10, 90) < 0.5, "cheap shortfalls: buy less than the median"

**2. A rule of thumb, and where it breaks**

The lecture found the bill close to ((short + long) / 2) × MAE × hours. Write `rule_of_thumb(schedule)` returning both the exact cost and the rule-of-thumb estimate (MEUR). Test it on the regression, and on the regression with a **bias** of +100 MW and −100 MW added. Why does the rule fail for biased schedules?

In [4]:
def rule_of_thumb(schedule, short=60.0, long=40.0) -> dict:
    """Exact imbalance cost and the rule-of-thumb estimate, both in MEUR."""
    ### BEGIN SOLUTION
    mae = float((schedule.loc[ACTUAL.index] - ACTUAL).abs().mean())
    return {"exact_MEUR": imbalance_cost(schedule, short, long) / 1e6,
            "rule_of_thumb_MEUR": (short + long) / 2 * mae * len(ACTUAL) / 1e6}
    ### END SOLUTION

In [5]:
thumb = pd.DataFrame({name: rule_of_thumb(REGRESSION + shift) for name, shift in
                      {"regression": 0, "regression + 100 MW": 100, "regression - 100 MW": -100}.items()}).T
thumb["error_%"] = 100 * (thumb.rule_of_thumb_MEUR / thumb.exact_MEUR - 1)
assert abs(thumb.loc["regression", "error_%"]) < 2
assert thumb.loc["regression - 100 MW", "exact_MEUR"] > thumb.loc["regression + 100 MW", "exact_MEUR"]
thumb.round(2)

                     exact_MEUR  rule_of_thumb_MEUR  error_%
regression                32.09               32.22     0.39
regression + 100 MW       40.81               49.69    21.78
regression - 100 MW       58.63               50.00   -14.72

### Core

**3. Steeper shortfall penalties**

Suppose shortfalls cost 150 €/MWh and surpluses 40 €/MWh. Using the quantiles of `ERRORS_2024`, compute the 2025 cost of buying the regression plus the α-quantile for α in [0.5, 0.6, …, 0.95]. Where is the minimum, how does it compare with `best_quantile(150, 40)`, and how much does it save against the point forecast? Write `steep_penalties()`.

In [6]:
def steep_penalties(alphas=np.round(np.arange(0.5, 0.951, 0.05), 2)) -> pd.Series:
    """2025 imbalance cost (MEUR) of buying the regression + the alpha-quantile of 2024 errors, short=150, long=40."""
    ### BEGIN SOLUTION
    return pd.Series({a: imbalance_cost(REGRESSION + np.quantile(ERRORS_2024, a), 150, 40) / 1e6 for a in alphas},
                     name="cost_MEUR").rename_axis("alpha")
    ### END SOLUTION

In [7]:
steep = steep_penalties()
point = imbalance_cost(REGRESSION, 150, 40) / 1e6
assert abs(steep.idxmin() - best_quantile(150, 40)) <= 0.1, "the minimum lies near the newsvendor quantile"
assert steep.min() < point
print(f"point forecast {point:.2f} MEUR; best alpha on 2025 {steep.idxmin()} -> {steep.min():.2f} MEUR; "
      f"theory alpha* = {best_quantile(150, 40):.3f}")
steep.round(2)

point forecast 60.52 MEUR; best alpha on 2025 0.8 -> 43.94 MEUR; theory alpha* = 0.789


alpha
0.50    66.95
0.55    60.89
0.60    55.60
0.65    51.42
0.70    47.82
0.75    45.35
0.80    43.94
0.85    44.06
0.90    46.80
0.95    55.75
Name: cost_MEUR, dtype: float64

**4. How demanding should the battery be?**

The lecture's battery traded whenever 0.85 × the forecast dear price exceeded the forecast cheap price. Require a **margin**: trade only if that difference exceeds m €/MWh, for m in [−5, 0, 5, 10, 20]. Use the tie-broken fundamental forecast. Report profit and trading days, and compare with perfect foresight at the same margins. Write `margin_sweep(margins)`.

In [8]:
FUNDAMENTAL = pd.Series(merit_order_price(REGRESSION - SOLAR.solar_forecast), index=df.index).loc["2025"]
TIE_BROKEN = FUNDAMENTAL + 1e-6 * pd.Series(HOUR_AVG_2024.loc[P2025.index.hour].to_numpy(), index=P2025.index)


def margin_sweep(margins=(-5, 0, 5, 10, 20)) -> pd.DataFrame:
    """Profit (MEUR) and trading days for each trading margin, forecast-driven and with perfect foresight."""
    ### BEGIN SOLUTION
    rows = {}
    for m in margins:
        f_profit, f_days = battery_profit(TIE_BROKEN, margin=m)
        p_profit, p_days = battery_profit(P2025, margin=m)
        rows[m] = {"forecast_profit_MEUR": f_profit / 1e6, "forecast_days": f_days,
                   "perfect_profit_MEUR": p_profit / 1e6, "perfect_days": p_days}
    return pd.DataFrame(rows).T.rename_axis("margin_EUR_per_MWh")
    ### END SOLUTION

In [9]:
sweep = margin_sweep()
assert np.isclose(sweep.loc[0, "forecast_profit_MEUR"], 1.29, atol=0.01), "the lecture's 1.29 MEUR at margin 0"
assert sweep.forecast_days.is_monotonic_decreasing
sweep.round(2)

                    forecast_profit_MEUR  forecast_days  perfect_profit_MEUR  \
margin_EUR_per_MWh                                                             
-5                                  1.27          221.0                 2.04   
 0                                  1.29          199.0                 2.10   
 5                                  1.29          190.0                 2.03   
 10                                 1.07          120.0                 1.91   
 20                                 0.96           97.0                 1.55   

                    perfect_days  
margin_EUR_per_MWh                
-5                         355.0  
 0                         287.0  
 5                         221.0  
 10                        181.0  
 20                        117.0  

**5. Interpretation**

Why does the rule of thumb fail for biased schedules, and in which direction? How did the steeper penalty move the best quantile, and was the theory right? What does the margin sweep say about how a battery should use an imperfect forecast?

<!-- BEGIN SOLUTION -->
The rule of thumb is nearly exact for the regression (0.4 % off) because its errors are about evenly split between short and long hours, so each MWh of error costs about the average penalty of 50 €. A biased schedule breaks that assumption. Buying 100 MW too much makes most hours long, and long hours cost only 40 €/MWh: the rule over-estimates the bill by 22 %. Buying 100 MW too little makes most hours short, at 60 €/MWh: the rule under-estimates by 15 %. And the two biases are not equally bad — under-buying costs 58.6 M€, over-buying 40.8 M€ — which is the asymmetry the newsvendor quantile exploits.

With shortfalls at 150 €/MWh the theory moves the best quantile to α* = 150 / 190 = 0.789, and 2025 agrees: the cost is lowest at α = 0.80 (43.9 M€), against 60.5 M€ for the point forecast — 27 % saved by buying more than the forecast. The steeper the asymmetry, the more the quantile is worth.

The margin sweep shows how to use an imperfect forecast: trade when the forecast says the spread is worth it, but not against it. With a margin of 0–5 €/MWh the battery earns 1.29 M€; a negative margin (trading on days the forecast itself calls unprofitable) earns a little less, and demanding 10 or 20 €/MWh skips too many profitable days (1.07 and 0.96 M€). Perfect foresight peaks at a margin of 0 as well — with a perfect forecast any positive spread is worth taking — while an imperfect forecast tolerates a small margin as a cushion against its own errors.
<!-- END SOLUTION -->

### Challenge

**6. What is a better solar forecast worth to the battery?**

Scale the solar forecast error: solar forecast_s = solar + s · (solar forecast − solar), for s in [0, 0.5, 1, 1.5]. For each s, build the fundamental price forecast from `REGRESSION` and the scaled solar forecast, break ties as in task 4, and report the price-forecast MAE for 2025 and the battery profit (margin 0). Write `solar_value(scales)`.

In [10]:
def solar_value(scales=(0, 0.5, 1, 1.5)) -> pd.DataFrame:
    """Price-forecast MAE (EUR/MWh) and battery profit (MEUR) for scaled solar-forecast errors."""
    ### BEGIN SOLUTION
    rows = {}
    tie = 1e-6 * pd.Series(HOUR_AVG_2024.loc[P2025.index.hour].to_numpy(), index=P2025.index)
    for s in scales:
        solar_fc = SOLAR.solar + s * (SOLAR.solar_forecast - SOLAR.solar)
        price_fc = pd.Series(merit_order_price(REGRESSION - solar_fc), index=df.index).loc["2025"] + tie
        rows[s] = {"solar_MAE_daylight_MW": float((solar_fc - SOLAR.solar).loc["2025"][SOLAR.solar.loc["2025"] > 1].abs().mean()),
                   "price_MAE": float((price_fc - P2025).abs().mean()),
                   "battery_profit_MEUR": battery_profit(price_fc)[0] / 1e6}
    return pd.DataFrame(rows).T.rename_axis("solar_error_scale")
    ### END SOLUTION

In [11]:
value = solar_value()
assert np.isclose(value.loc[1, "battery_profit_MEUR"], 1.29, atol=0.01)
assert value.loc[0, "battery_profit_MEUR"] > value.loc[1.5, "battery_profit_MEUR"]
value.round(3)

                   solar_MAE_daylight_MW  price_MAE  battery_profit_MEUR
solar_error_scale                                                       
0.0                                0.000      5.905                1.385
0.5                               35.731      6.019                1.340
1.0                               71.461      6.421                1.286
1.5                              107.192      6.940                1.224

**7. Interpretation**

How much would a perfect solar forecast be worth to this battery, and how much does a 50 % worse one cost? Why does even a perfect solar forecast leave the battery far below perfect foresight?

<!-- BEGIN SOLUTION -->
A perfect solar forecast would raise the battery's profit from 1.29 to 1.39 M€ — about 0.10 M€ a year; a solar forecast 50 % worse than today's would cost about 0.06 M€. That is small, and the price forecast's MAE shows why: even with perfect solar it only improves from 6.42 to 5.91 €/MWh. Where does the rest of the gap to perfect foresight (2.10 M€) come from? Making the demand forecast perfect as well brings the price MAE down to 3.98 €/MWh and the profit to 1.59 M€ — another 0.20 M€, twice what perfect solar adds: lesson 2's 74 MW demand error moves residual demand across the stack's blocks, and the steps turn small errors near a block boundary into large price errors. The last 0.51 M€ belongs to the price deviations that no merit order explains, which only perfect foresight knows. The value of improving one input depends on how much of the final error it causes — here the solar forecast is a minor part, and money for a better forecast would be better spent on demand or on modelling the price deviations.
<!-- END SOLUTION -->